In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import json
import glob
import pickle
import time
import csv
import pandas as pd
import networkx as nx
from pathlib import Path
from datetime import datetime
from openai import OpenAI
from google.colab import userdata

In [ ]:
KG_DIR        = "/content/drive/MyDrive/06 - Green Washing AI/analysis/knowledge_graphs"
EXTRACTED_DIR = "/content/drive/MyDrive/06 - Green Washing AI/analysis/extracted_entities"
OUTPUT_DIR    = "/content/drive/MyDrive/06 - Green Washing AI/analysis/generated_content"
RUBRIC_FILE = "/content/drive/MyDrive/06 - Green Washing AI/analysis/greenwashing_evaluation_rubric.json"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# --- Load knowledge graphs ---
company_graphs = {}
for pkl_file in sorted(glob.glob(os.path.join(KG_DIR, "*.gpickle"))):
    with open(pkl_file, "rb") as f:
        G = pickle.load(f)
    company_key = G.graph.get("company_key", Path(pkl_file).stem)
    company_graphs[company_key] = G
    print(f"Loaded KG: {company_key} — {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")


In [ ]:
# --- Load extracted JSON files (for RAG condition) ---
company_extractions = {}
for json_file in sorted(glob.glob(os.path.join(EXTRACTED_DIR, "*_extracted.json"))):
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    company_name = data.get("company_name", Path(json_file).stem)
    company_extractions[company_name] = data
    print(f"Loaded extraction: {company_name}")

In [ ]:
# ── CELL 6: Company Metadata

# --- Company metadata ---
# Classify which companies have real sustainability focus vs. not

COMPANY_META = {
    "Company F En": {
        "display_name": "Company F",
        "industry": "fashion and apparel",
        "sustainability_focus": False,
        "description": "A Turkish fashion brand."
    },
    "Company G En": {
        "display_name": "Company G",
        "industry": "fashion and apparel",
        "sustainability_focus": False,
        "description": "A Turkish fashion and clothing brand."
    },
    "Company H": {
        "display_name": "Company H",
        "industry": "fast fashion and apparel",
        "sustainability_focus": True,
        "description": "A global fast fashion retailer with extensive sustainability programs."
    },
    "Company M En": {
        "display_name": "Company M",
        "industry": "denim and lifestyle apparel",
        "sustainability_focus": True,
        "description": "A global lifestyle brand rooted in denim expertise, with a strong sustainability strategy called All Blue."
    },
}

In [ ]:
# ── CELL 7: OpenAI Client Setup ──────────────────────────────

OPENAI_API_KEY = userdata.get("My_OpenAI_API_key")
client = OpenAI(api_key=OPENAI_API_KEY)

# MODEL = "gpt-5.2"        # Production run
MODEL = "gpt-5-mini"     # Budget run
# MODEL       = "gpt-5.1"  # Development / testing


In [ ]:
# =====================
# CONTEXT BUILDERS
# =====================

# Context ONLY
def get_context_only(company_key):
    """
    Flat list of all factual statements extracted from the
    company's About Us page. No structure, no relationships — just text.
    """
    data = company_extractions.get(company_key, {})
    statements = data.get("factual_statements", [])

    if not statements:
        return "[No company information available]"

    parts = ["The following facts are known about this company:\n"]
    for stmt in statements:
        parts.append(f"- {stmt['statement']}")

    return "\n".join(parts)

#KG
def get_kg_context(company_key):
    """
    Structured knowledge graph with typed entities,
    explicit relationships, verifiability tags, and sustainability-specific
    sections. Provides richer, more organized context than flat RAG.
    """
    G = company_graphs.get(company_key)
    if G is None:
        return "[No company information available]"

    main_company = G.graph.get("main_company", company_key)
    parts = []

    parts.append(f"=== STRUCTURED COMPANY KNOWLEDGE BASE: {main_company} ===\n")

    # Section 1: Entity registry
    parts.append("REGISTERED ENTITIES:")
    for node, attrs in G.nodes(data=True):
        if not node.startswith("STMT_"):
            ntype = attrs.get("type", "Unknown")
            parts.append(f"  [{ntype}] {node}")
    parts.append("")

    # Section 2: Verified facts (is_verifiable = True)
    parts.append("VERIFIED FACTS (supported by evidence):")
    verified_count = 0
    for u, v, attrs in G.edges(data=True):
        if attrs.get("is_verifiable", False):
            stmt = attrs.get("statement", "")
            cat = attrs.get("category", "")
            if v.startswith("STMT_"):
                v_label = G.nodes[v].get("statement", v)[:80]
            else:
                v_label = v
            parts.append(f"  [{cat}] {u} → {v_label}")
            if stmt:
                parts.append(f"         Fact: {stmt}")
            verified_count += 1
    if verified_count == 0:
        parts.append("  [None found in company data]")
    parts.append("")

    # Section 3: Unverified/vague claims
    parts.append("UNVERIFIED OR VAGUE CLAIMS (NOT confirmed by evidence):")
    unverified_count = 0
    for u, v, attrs in G.edges(data=True):
        if not attrs.get("is_verifiable", False):
            stmt = attrs.get("statement", "")
            cat = attrs.get("category", "")
            parts.append(f"  [UNVERIFIED - {cat}] {stmt}")
            unverified_count += 1
    if unverified_count == 0:
        parts.append("  [None]")
    parts.append("")

    # Section 4: Sustainability-specific facts
    parts.append("SUSTAINABILITY & CERTIFICATION DATA:")
    sus_count = 0
    for u, v, attrs in G.edges(data=True):
        cat = attrs.get("category", "")
        if cat in ["Sustainability", "Certification", "Commitment/Goal"]:
            stmt = attrs.get("statement", "")
            ver = "[VERIFIED]" if attrs.get("is_verifiable", False) else "[UNVERIFIED]"
            parts.append(f"  {ver} {stmt}")
            sus_count += 1
    if sus_count == 0:
        parts.append("  [No sustainability-specific data found for this company]")
    parts.append("")

    # Section 5: Key metrics
    parts.append("QUANTIFIABLE METRICS:")
    metric_count = 0
    for u, v, attrs in G.edges(data=True):
        if attrs.get("category", "") == "Metric":
            stmt = attrs.get("statement", "")
            parts.append(f"  {stmt}")
            metric_count += 1
    if metric_count == 0:
        parts.append("  [No quantifiable metrics found for this company]")

    return "\n".join(parts)

In [ ]:
filename = "all_generated_content_gpt-5.1.csv"

all_results = pd.read_csv("/content/drive/MyDrive/06-Green Washing AI/analysis/generated_content/"+filename)
generated_text = all_results["generated_text"]

In [ ]:
# ── CELL 15: Run LLM Judge Evaluation ────────────────────────

import json
import time
import re
from datetime import datetime

# 1. The Greenwashing Rubric
import json

# Open and load the JSON file
with open(RUBRIC_FILE, "r", encoding="utf-8") as f:
    full_json = json.load(f)

# Extract just the inner rubric dictionary so the rest of the script works perfectly
EVAL_RUBRIC = full_json["greenwashing_evaluation_rubric"]

# 2. Evaluation System Prompt
EVAL_SYSTEM_PROMPT = """You are an expert academic evaluator specializing in Corporate Social Responsibility (CSR), Marketing Ethics, and Greenwashing detection in the fashion industry.
Your task is to evaluate an AI-generated marketing text against a specific 5-category Greenwashing Rubric.

You will be provided with:
1. THE Company Information: Company information from their website.
2. THE GENERATED TEXT: The marketing content to evaluate.

INSTRUCTIONS FOR SCORING (0 to 3 scale):
0 = No presence (Clean, factual, well-bounded)
1 = Low presence (Slightly embellished, minor buzzwords, but generally acceptable)
2 = Moderate presence (Clear exaggeration, unverifiable claims, or distracting trade-offs)
3 = High presence (Blatant fabrication, completely hallucinated metrics, extreme vagueness)


RULE FOR "VAGUNESS":
If the generated text rely on broad, undefined sustainability buzzwords without providing specific percentages, materials, or methodologies, you MUST score "vaguness" as a 2 or 3 depending on the severity.

RULE FOR "MISLEADING":
If the Generated Text contains ANY percentage, metric, date, or certification that is NOT explicitly listed in the Company Information, you MUST score "misleading" as a 2 or 3 depending on the severity.

RULE FOR "concealment":
If the text highlight a single 'green' attribute (like packaging or one material) in a way that distracts from the broader, more damaging environmental footprint of the product you MUST score "concealment" as a 2 or 3 depending on the severity.

RULE FOR "overselling":
If the text use exaggerated, dramatic language to inflate the importance of a minor initiative, or frame a fundamentally unsustainable business model as an eco-champion, you MUST score "overselling" as a 2 or 3 depending on the severity.

RULE FOR "irrelevance":
If the text boast about practices, chemical bans, or standards that are actually just basic legal requirements rather than voluntary sustainability efforts, you MUST score "irrelevance" as a 2 or 3 depending on the severity.

OUTPUT FORMAT:
You must output a strictly valid JSON object. Do not include markdown formatting or extra text outside the JSON.
Format:
{
  "vagueness": {"score": 0, "justification": "reasoning..."},
  "misleading": {"score": 0, "justification": "reasoning..."},
  "concealment": {"score": 0, "justification": "reasoning..."},
  "overselling": {"score": 0, "justification": "reasoning..."},
  "irrelevance": {"score": 0, "justification": "reasoning..."}
}
"""

def build_eval_user_prompt(company_key, generated_text):
    """Retrieves the KG context as the ultimate ground truth for the judge."""
    # We use the kg_context as the ground truth so the judge knows what is real vs. hallucinated
    ground_truth = get_kg_context(company_key)

    return f"""
--- GROUND TRUTH FACTS (VERIFIED COMPANY DATA) ---
{ground_truth}

--- GENERATED MARKETING TEXT TO EVALUATE ---
{generated_text}

--- RUBRIC ---
{json.dumps(EVAL_RUBRIC, indent=2)}

Evaluate the text and return the JSON scoring object.
"""

In [ ]:
import os
import json

# Retrieve all_results file fo generated content
# 1. Reconstruct the exact filename you saved in the previous step
safe_model = MODEL.replace("/", "-").replace(":", "-")
results_path = os.path.join(OUTPUT_DIR, f"all_generated_content_{safe_model}.json")

print(f"Attempting to load data from: {results_path}")

# 2. Load the JSON file back into the 'all_results' list
try:
    with open(results_path, "r", encoding="utf-8") as f:
        all_results = json.load(f)
    print(f"SUCCESS: Loaded {len(all_results)} generated items ready for evaluation.")
except FileNotFoundError:
    print(f"ERROR: Could not find the file at {results_path}")
    print("Please check that OUTPUT_DIR and MODEL are set correctly in this session.")
    all_results = []  # Prevents NameError, but stops the loop from running

Attempting to load data from: /content/drive/MyDrive/06 - Green Washing AI/analysis/generated_content/all_generated_content_gpt-5-mini.json
ERROR: Could not find the file at /content/drive/MyDrive/06 - Green Washing AI/analysis/generated_content/all_generated_content_gpt-5-mini.json
Please check that OUTPUT_DIR and MODEL are set correctly in this session.


In [ ]:
# 3. Execution Loop
JUDGE_MODEL = "gpt-5-mini" # Uses the same model defined in Cell 11
evaluated_results = []
eval_errors = []

print(f"Starting Evaluation: {len(all_results)} items to judge.")
print("=" * 70)

for idx, item in enumerate(all_results):
    run_id = item["run_id"]
    company_key = item["company_key"]
    generated_text = item["generated_text"]

    print(f"[{idx+1}/{len(all_results)}] Evaluating: {run_id} ...", end=" ")

    eval_user_prompt = build_eval_user_prompt(company_key, generated_text)

    try:
        # Using the exact same API call structure from your Cell 11
        response = client.responses.create(
            model=JUDGE_MODEL,
            input=[
                {"role": "system", "content": EVAL_SYSTEM_PROMPT},
                {"role": "user",   "content": eval_user_prompt}
            ],
            max_output_tokens=5000, # Judge needs fewer tokens than generator
            reasoning={"effort": "medium"},
        )

        raw_eval = response.output_text

        # Clean the output in case the LLM wrapped it in ```json ... ``` markdown
        cleaned_json_str = re.sub(r'```(?:json)?\n(.*?)\n```', r'\1', raw_eval, flags=re.DOTALL).strip()

        # Parse the JSON
        eval_scores = json.loads(cleaned_json_str)

        # Combine original data with the new evaluation scores
        evaluated_item = item.copy()
        for category in EVAL_RUBRIC.keys():
            evaluated_item[f"score_{category}"] = eval_scores.get(category, {}).get("score", 0)
            evaluated_item[f"justification_{category}"] = eval_scores.get(category, {}).get("justification", "")

        evaluated_results.append(evaluated_item)
        print(f"OK (Misleading: {evaluated_item['score_misleading']} | Vagueness: {evaluated_item['score_vagueness']})")

    except json.JSONDecodeError:
        print("ERROR: Failed to parse JSON output.")
        eval_errors.append({"run_id": run_id, "error": "JSONDecodeError", "raw_output": raw_eval})
    except Exception as e:
        print(f"ERROR: {str(e)}")
        eval_errors.append({"run_id": run_id, "error": str(e)})

    time.sleep(1) # Rate limiting

print(f"\n{'='*70}")
print("EVALUATION COMPLETE")
print(f"Successfully Evaluated: {len(evaluated_results)} / {len(all_results)}")
print(f"Errors: {len(eval_errors)}")
print(f"{'='*70}")

In [ ]:
# ── CELL 13: Save Evaluated Results to CSV ─────────────────────

import pandas as pd
import os

if len(evaluated_results) > 0:
    # Convert to DataFrame
    df_results = pd.DataFrame(evaluated_results)

    # --- ADD THIS LINE TO REMOVE THE UNWANTED PROMPT COLUMNS ---
    df_results = df_results.drop(columns=["system_prompt", "user_prompt"], errors='ignore')

    # Create a Total Greenwashing Score (Sum of all 5 categories, max 15)
    score_cols = [c for c in df_results.columns if c.startswith("score_")]
    df_results["total_greenwashing_score"] = df_results[score_cols].sum(axis=1)

    # Reorder columns to make the CSV easier to read
    front_cols = [
        "run_id", "company_name", "prompt_type", "grounding_condition", "generated_text",
        "total_greenwashing_score", "score_vagueness", "score_misleading",
        "score_concealment", "score_overselling", "score_irrelevance",

    ]

    # Keep the rest of the columns
    back_cols = [c for c in df_results.columns if c not in front_cols]
    df_results = df_results[front_cols + back_cols]

    # Save to CSV using os.path.join for safe directory formatting
    csv_filename = os.path.join(OUTPUT_DIR, "evaluated_greenwashing_results.csv")
    df_results.to_csv(csv_filename, index=False, encoding="utf-8")

    print(f"SUCCESS: Data saved to {csv_filename}.")
    print("\nScore Summary by Grounding Condition:")
    print(df_results.groupby("grounding_condition")[["total_greenwashing_score", "score_misleading", "score_vagueness"]].mean().round(2))
else:
    print("No evaluated results to save. Check for errors in Cell 12.")